# JARVIS V6 — Persistent Colab Studio
Run the AI from source, keep learned state on Google Drive, chat first, then scale and train on GPU. The GitHub repository is the source code; checkpoint `.pt` files are the learned model state.

## 1) Clone the repository

In [11]:
REPO_URL = 'https://github.com/Aditya-0167/jarvis-the-future.git'
%cd /content
!rm -rf jarvis_v6
!git clone --depth 1 $REPO_URL jarvis_v6
%cd /content/jarvis_v6
!python -m pip install -q -r requirements.txt

/content
Cloning into 'jarvis_v6'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 84 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 121.89 KiB | 3.58 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/jarvis_v6


## 2) Mount Google Drive for persistent JARVIS state

In [10]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os
STATE_ROOT = Path('/content/drive/MyDrive/JARVIS_STATE')
STATE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['JARVIS_ROOT'] = str(STATE_ROOT)
print('Persistent state:', STATE_ROOT)

MessageError: Error: credential propagation was unsuccessful

## 3) Import the learned state from GitHub
Download the `jarvis_v6_state.zip` artifact from your completed GitHub Actions run. This archive should contain `checkpoints/*.pt` plus the state/data folders. Upload it once here.

In [4]:
from google.colab import files
import shutil, zipfile, tempfile
uploads = files.upload()
for name in uploads:
    src = Path('/content') / name
    if zipfile.is_zipfile(src):
        tmp = Path(tempfile.mkdtemp(prefix='jarvis_import_'))
        with zipfile.ZipFile(src) as z: z.extractall(tmp)
        roots = list(tmp.rglob('checkpoints'))
        bundle_root = roots[0].parent if roots else tmp
        for folder in ['data','checkpoints','generations','workspace']:
            candidate = bundle_root / folder
            if candidate.exists(): shutil.copytree(candidate, STATE_ROOT / folder, dirs_exist_ok=True)
        shutil.rmtree(tmp, ignore_errors=True)
        print('Imported:', name)
    else:
        print('Not a zip, skipping:', name)
    src.unlink(missing_ok=True)
print('State ready')

Saving JARVIS_v6_FIXED_VERIFIED.zip to JARVIS_v6_FIXED_VERIFIED.zip
Imported: JARVIS_v6_FIXED_VERIFIED.zip
State ready


In [5]:
from pathlib import Path
import shutil
import zipfile
import tempfile

src = Path("/content/jarvis-v6-run-1.zip")

print("Found upload:", src.exists())
print("Size (MB):", round(src.stat().st_size / (1024**2), 1))

with tempfile.TemporaryDirectory(prefix="jarvis_import_") as td:
    tmp = Path(td)

    with zipfile.ZipFile(src) as z:
        z.extractall(tmp)

    roots = list(tmp.rglob("checkpoints"))
    bundle_root = roots[0].parent if roots else tmp

    for folder in ["data", "checkpoints", "generations", "workspace"]:
        candidate = bundle_root / folder
        if candidate.exists():
            shutil.copytree(
                candidate,
                Path("/content/JARVIS_STATE") / folder,
                dirs_exist_ok=True
            )

print("JARVIS state imported successfully.")
print("State location: /content/JARVIS_STATE")

Found upload: True
Size (MB): 317.1
JARVIS state imported successfully.
State location: /content/JARVIS_STATE


## 4) Verify the checkpoint and GPU

In [12]:
%cd /content/jarvis_v6
!python main.py status
!python -c "import torch; print('torch', torch.__version__); print('cuda', torch.cuda.is_available()); print('device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"


/content/jarvis_v6
{
  "generation": 0,
  "training_steps": 0,
  "accepted_changes": 0,
  "best_eval_loss": null,
  "profile": "local",
  "architecture": {
    "vocab_size": 256,
    "d_model": 192,
    "n_heads": 6,
    "n_layers": 6,
    "experts": 2,
    "top_k_experts": 2,
    "dropout": 0.05,
    "block_size": 384
  },
  "parameters": 6267852,
  "lineage": [],
  "web_pages": 0,
  "last_web_cycle": null,
  "last_error": null,
  "runtime": {
    "running": false,
    "operation": null,
    "step": 0,
    "total": 0,
    "loss": null
  },
  "corpus_characters": 240,
  "memory_records": 1,
  "experiment_records": 0,
  "device": "cuda",
  "checkpoint": "/content/jarvis_v6/checkpoints/generation_000000.pt"
}
torch 2.11.0+cu128
cuda True
device Tesla T4


In [14]:
from pathlib import Path

print("ZIP files currently in Colab:")
for f in Path("/content").glob("*.zip"):
    print(f.name, round(f.stat().st_size / (1024**2), 1), "MB")

ZIP files currently in Colab:
jarvis-v6-run-1.zip 317.1 MB


## 5) CHAT FIRST
Run the next cell. It launches JARVIS Studio with a chat panel and a live monitor. The Studio is the place where you can talk to the current checkpoint and watch training/evolution state.

In [15]:
!python main.py studio

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://70483a9048af987d62.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
stop
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 3564, in block_thread
    time.sleep(0.1)
    ~~~~~~~~~~^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/jarvis_v6/main.py", line 60, in <module>
    main()
    ~~~~^^
  File "/content/jarvis_v6/main.py", line 53, in main
    elif args.cmd == "studio": launch_studio(system, share=not args.no_share); return
                               ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^

## 6) After you have tested chat, scale the model to the Colab profile
This creates a new generation. Compatible learned tensors are copied; newly created capacity is freshly initialized.

In [16]:
%cd /content/jarvis_v6
!python main.py profile colab
!python main.py status

/content/jarvis_v6
{
  "generation": 1,
  "training_steps": 0,
  "accepted_changes": 0,
  "best_eval_loss": null,
  "profile": "colab",
  "architecture": {
    "vocab_size": 256,
    "d_model": 384,
    "n_heads": 8,
    "n_layers": 12,
    "experts": 6,
    "top_k_experts": 2,
    "dropout": 0.05,
    "block_size": 768
  },
  "parameters": 134864328,
  "lineage": [],
  "web_pages": 0,
  "last_web_cycle": null,
  "last_error": null,
  "runtime": {
    "running": false,
    "operation": null,
    "step": 0,
    "total": 0,
    "loss": null
  }
}
{
  "generation": 1,
  "training_steps": 0,
  "accepted_changes": 0,
  "best_eval_loss": null,
  "profile": "colab",
  "architecture": {
    "vocab_size": 256,
    "d_model": 384,
    "n_heads": 8,
    "n_layers": 12,
    "experts": 6,
    "top_k_experts": 2,
    "dropout": 0.05,
    "block_size": 768
  },
  "parameters": 134864328,
  "lineage": [],
  "web_pages": 0,
  "last_web_cycle": null,
  "last_error": null,
  "runtime": {
    "running": f

In [17]:
!python main.py crawl

{
  "pages": [
    {
      "url": "https://en.wikipedia.org/wiki/Artificial_intelligence",
      "depth": 0,
      "text": "",
      "error": "PermissionError('robots.txt disallows or is unavailable for automatic crawl: https://en.wikipedia.org/wiki/Artificial_intelligence')"
    },
    {
      "url": "https://www.python.org/doc/",
      "depth": 0,
      "text": "Our Documentation | Python.org\nNotice:\nThis page displays a fallback because interactive scripts did not run. Possible causes include disabled JavaScript or failure to load scripts or stylesheets.\nSkip to content\n▼\nClose\nPython\nPSF\nDocs\nPyPI\nJobs\nCommunity\n▲\nThe Python Network\nDonate\n≡\nMenu\nSearch This Site\nGO\nA\nA\nSmaller\nLarger\nReset\nSocialize\nLinkedIn\nMastodon\nChat on IRC\nTwitter\nAbout\nApplications\nQuotes\nGetting Started\nHelp\nDownloads\nAll releases\nSource code\nWindows\nmacOS\nAndroid\niOS\nOther Platforms\nLicense\nAlternative Implementations\nDocumentation\nDocs\nAudio/Visual Talks\nBeg

In [18]:
!python main.py status

{
  "generation": 1,
  "training_steps": 0,
  "accepted_changes": 0,
  "best_eval_loss": null,
  "profile": "colab",
  "architecture": {
    "vocab_size": 256,
    "d_model": 384,
    "n_heads": 8,
    "n_layers": 12,
    "experts": 6,
    "top_k_experts": 2,
    "dropout": 0.05,
    "block_size": 768
  },
  "parameters": 134864328,
  "lineage": [],
  "web_pages": 5,
  "last_web_cycle": 1789655125.7585475,
  "last_error": "1 crawl errors; inspect memory log",
  "runtime": {
    "running": false,
    "operation": null,
    "step": 0,
    "total": 0,
    "loss": null
  },
  "web_pages_session": 5,
  "corpus_characters": 21968,
  "memory_records": 10,
  "experiment_records": 0,
  "device": "cuda",
  "checkpoint": "/content/jarvis_v6/checkpoints/generation_000001.pt"
}


## 7) Small GPU sanity training run

In [19]:
!python main.py train --steps 50
!python main.py benchmark

    step 25/50 loss=3.6243
    step 50/50 loss=3.4485
{
  "loss_before": 5.629691998163859,
  "loss_after": 3.6263696352640786,
  "steps": 50,
  "seconds": 28.156933784484863,
  "device": "cuda",
  "parameters": 134864328
}
{
  "validation_cross_entropy": 3.6180042823155723,
  "perplexity": 37.26312688391351,
  "bytes_in_corpus": 22092,
  "device": "cuda",
  "note": "Lower validation loss/perplexity is better. Compare only across comparable data/model conditions."
}


## 8) Architecture evolution test

In [20]:
!python main.py evolve
!python main.py status
!python main.py manifest

    step 25/40 loss=3.2593
    step 25/40 loss=3.2505
{
  "accepted": true,
  "trial_id": "b4f222818d08",
  "base_loss": 3.6371081272761026,
  "best_loss": 3.0603533585866294,
  "action": "narrow_dropout",
  "model_cfg": {
    "vocab_size": 256,
    "d_model": 384,
    "n_heads": 8,
    "n_layers": 12,
    "experts": 6,
    "top_k_experts": 2,
    "dropout": 0.025,
    "block_size": 768
  },
  "candidate_losses": [
    {
      "action": "narrow_dropout",
      "loss": 3.0603533585866294,
      "parameters": 134864328
    },
    {
      "action": "increase_top_k",
      "loss": 3.199031392733256,
      "parameters": 134864328
    },
    {
      "action": "keep",
      "loss": 3.6371081272761026,
      "parameters": 134864328
    }
  ]
}
{
  "generation": 2,
  "training_steps": 50,
  "accepted_changes": 1,
  "best_eval_loss": 3.0603533585866294,
  "profile": "colab",
  "architecture": {
    "vocab_size": 256,
    "d_model": 384,
    "n_heads": 8,
    "n_layers": 12,
    "experts": 6,
   

## 9) Public web refresh + learning

In [21]:
!python main.py crawl
!python main.py train --steps 100
!python main.py benchmark

{
  "pages": [
    {
      "url": "https://en.wikipedia.org/wiki/Artificial_intelligence",
      "depth": 0,
      "text": "",
      "error": "PermissionError('robots.txt disallows or is unavailable for automatic crawl: https://en.wikipedia.org/wiki/Artificial_intelligence')"
    },
    {
      "url": "https://www.python.org/doc/",
      "depth": 0,
      "text": "Our Documentation | Python.org\nNotice:\nThis page displays a fallback because interactive scripts did not run. Possible causes include disabled JavaScript or failure to load scripts or stylesheets.\nSkip to content\n▼\nClose\nPython\nPSF\nDocs\nPyPI\nJobs\nCommunity\n▲\nThe Python Network\nDonate\n≡\nMenu\nSearch This Site\nGO\nA\nA\nSmaller\nLarger\nReset\nSocialize\nLinkedIn\nMastodon\nChat on IRC\nTwitter\nAbout\nApplications\nQuotes\nGetting Started\nHelp\nDownloads\nAll releases\nSource code\nWindows\nmacOS\nAndroid\niOS\nOther Platforms\nLicense\nAlternative Implementations\nDocumentation\nDocs\nAudio/Visual Talks\nBeg

## 10) Longer autonomous run
This lets the internal policy choose among training, web refresh, evolution, and benchmarking within the configured research sandbox.

In [22]:
!python main.py autonomous --cycles 30

=== JARVIS V6 CYCLE 1 ===
    step 25/40 loss=2.4033
    step 25/40 loss=2.4102
DEVELOP {'generation': 3, 'selected_action': 'evolve', 'observations': {'training_loss_after': None, 'evolution_action': 'increase_top_k', 'accepted': True, 'web_characters_added': 0, 'intrinsic_improvement': True}, 'next_actions': ['train', 'evaluate', 'continue_with_new_architecture', 'refresh_public_web_corpus', 'checkpoint'], 'policy': {'train': 1.003, 'evolve': 1.12, 'web': 1.003, 'benchmark': 1.003}, 'note': 'Machine-generated control data, not a claim of consciousness or AGI.'}
=== JARVIS V6 CYCLE 2 ===
    step 25/40 loss=1.9230
    step 25/40 loss=2.1364
DEVELOP {'generation': 4, 'selected_action': 'evolve', 'observations': {'training_loss_after': None, 'evolution_action': 'add_expert', 'accepted': True, 'web_characters_added': 0, 'intrinsic_improvement': True}, 'next_actions': ['train', 'evaluate', 'continue_with_new_architecture', 'refresh_public_web_corpus', 'checkpoint'], 'policy': {'train': 1.

In [23]:
!python main.py studio

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://b96e40029eaacb969d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 3564, in block_thread
    time.sleep(0.1)
    ~~~~~~~~~~^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/jarvis_v6/main.py", line 60, in <module>
    main()
    ~~~~^^
  File "/content/jarvis_v6/main.py", line 53, in main
    elif args.cmd == "studio": launch_studio(system, share=not args.no_share); return
                               ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [24]:
from pathlib import Path
import shutil
from google.colab import files

STATE = Path("/content/JARVIS_STATE")
OUT = Path("/content/JARVIS_V6_BACKUP")

if OUT.exists():
    shutil.rmtree(OUT)

shutil.copytree(STATE, OUT)

archive = shutil.make_archive(
    "/content/JARVIS_V10_BACKUP",
    "zip",
    "/content/JARVIS_V10_BACKUP"
)

print("Backup ready:")
print(archive)
print("Size:", round(Path(archive).stat().st_size / (1024**2), 1), "MB")

files.download(archive)

Backup ready:
/content/JARVIS_V6_BACKUP.zip
Size: 158.6 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
%cd /content/jarvis_v6
!python main.py status
!python main.py manifest

/content/jarvis_v6
{
  "generation": 10,
  "training_steps": 220,
  "accepted_changes": 9,
  "best_eval_loss": 1.8307004968325298,
  "profile": "colab",
  "architecture": {
    "vocab_size": 256,
    "d_model": 384,
    "n_heads": 8,
    "n_layers": 12,
    "experts": 7,
    "top_k_experts": 4,
    "dropout": 0.0,
    "block_size": 1280
  },
  "parameters": 156144084,
  "lineage": [
    {
      "generation": 2,
      "action": "narrow_dropout",
      "accepted": true,
      "parameters": 134864328
    },
    {
      "generation": 3,
      "action": "increase_top_k",
      "accepted": true,
      "parameters": 134864328
    },
    {
      "generation": 4,
      "action": "add_expert",
      "accepted": true,
      "parameters": 156144084
    },
    {
      "generation": 5,
      "action": "widen_context",
      "accepted": true,
      "parameters": 156144084
    },
    {
      "generation": 6,
      "action": "narrow_dropout",
      "accepted": true,
      "parameters": 156144084
    },

In [26]:
from pathlib import Path
import shutil
from google.colab import files

# The ACTUAL latest checkpoint reported by JARVIS
latest = Path("/content/jarvis_v6/checkpoints/generation_000010.pt")

if not latest.exists():
    raise FileNotFoundError(f"Latest checkpoint not found: {latest}")

out = Path("/content/JARVIS_GENERATION_10")

if out.exists():
    shutil.rmtree(out)

(out / "checkpoints").mkdir(parents=True)

# Copy the actual latest learned model
shutil.copy2(
    latest,
    out / "checkpoints" / "generation_000010.pt"
)

# Copy useful persistent research state if present
for folder in ["data", "generations", "workspace"]:
    src = Path("/content/JARVIS_STATE") / folder
    if src.exists():
        shutil.copytree(
            src,
            out / folder,
            dirs_exist_ok=True
        )

# Copy the project config if present
config = Path("/content/jarvis_v6/config.json")
if config.exists():
    shutil.copy2(config, out / "config.json")

archive = shutil.make_archive(
    "/content/JARVIS_GENERATION_10",
    "zip",
    out
)

size_mb = Path(archive).stat().st_size / (1024**2)

print("LATEST JARVIS BACKUP READY")
print("Generation: 10")
print("Checkpoint:", latest)
print("Checkpoint size MB:", round(latest.stat().st_size / (1024**2), 1))
print("Backup:", archive)
print("Backup size MB:", round(size_mb, 1))

files.download(archive)

LATEST JARVIS BACKUP READY
Generation: 10
Checkpoint: /content/jarvis_v6/checkpoints/generation_000010.pt
Checkpoint size MB: 595.9
Backup: /content/JARVIS_GENERATION_10.zip
Backup size MB: 552.9


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
from pathlib import Path

state = Path("/content/JARVIS_STATE")

print("=== JARVIS DATA FILES ===")

for p in sorted(state.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / (1024**2)
        if p.suffix.lower() in {".txt", ".json", ".jsonl", ".csv", ".md"}:
            print(f"{p.relative_to(state)}  |  {size_mb:.2f} MB")

=== JARVIS DATA FILES ===
checkpoints/LATEST.json  |  0.00 MB
generations/generation_000001/result.json  |  0.00 MB
generations/generation_000002/result.json  |  0.00 MB
generations/generation_000003/result.json  |  0.00 MB
generations/generation_000004/result.json  |  0.00 MB
workspace/development_plan.json  |  0.00 MB
workspace/events.jsonl  |  0.01 MB
workspace/experiments.jsonl  |  0.00 MB
workspace/intrinsic_policy.json  |  0.00 MB
workspace/policy.json  |  0.00 MB
workspace/state.json  |  0.00 MB


In [ ]:
%cd /content/jarvis_v6

from pathlib import Path
import random
import subprocess
import textwrap

random.seed(1701)

# ------------------------------------------------------------
# 1. Build a broad learning curriculum
# ------------------------------------------------------------

out = Path("/content/JARVIS_CORE_CURRICULUM.txt")

rows = []

def add(q, a, domain):
    rows.append(
        f"[{domain}]\n"
        f"Question: {q}\n"
        f"Answer: {a}\n"
        f"Explanation: {a}\n\n"
    )

# ---------------- MATH ----------------

for _ in range(12000):
    a = random.randint(0, 999)
    b = random.randint(0, 999)
    op = random.choice(["+", "-", "*"])

    if op == "+":
        ans = a + b
    elif op == "-":
        ans = a - b
    else:
        ans = a * b

    add(
        f"What is {a} {op} {b}?",
        str(ans),
        "MATH"
    )

for _ in range(5000):
    x = random.randint(2, 100)
    k = random.randint(2, 20)
    b = random.randint(0, 100)
    c = k * x + b

    add(
        f"Solve for x: {k}x + {b} = {c}.",
        str(x),
        "ALGEBRA"
    )

for _ in range(3000):
    n = random.randint(1, 20)
    seq = [n + i * random.randint(2, 9) for i in range(4)]
    step = seq[1] - seq[0]

    add(
        f"What comes next in the sequence {seq[0]}, {seq[1]}, {seq[2]}, {seq[3]}?",
        str(seq[3] + step),
        "SEQUENCES"
    )

# ---------------- SCIENCE ----------------

science = [
    ("What is gravity?",
     "Gravity is the attractive interaction between masses. Near Earth it accelerates falling objects downward at about 9.81 meters per second squared."),
    ("What is Newton's first law?",
     "An object remains at rest or moves at constant velocity unless acted on by a net external force."),
    ("What is Newton's second law?",
     "The net force on an object equals its mass multiplied by its acceleration: F = ma."),
    ("What is energy?",
     "Energy is the capacity to cause physical change or perform work."),
    ("What is kinetic energy?",
     "Kinetic energy is the energy associated with motion. For a mass m moving at speed v, it is one half m v squared."),
    ("What is an atom?",
     "An atom is the smallest unit of an element that retains that element's chemical identity."),
    ("What is a molecule?",
     "A molecule is a group of two or more atoms held together by chemical bonds."),
    ("What is the periodic table?",
     "The periodic table organizes chemical elements by atomic number and recurring chemical properties."),
    ("What is DNA?",
     "DNA is the molecule that stores hereditary genetic information in living organisms."),
    ("What is a cell?",
     "A cell is the basic structural and functional unit of life."),
    ("What is photosynthesis?",
     "Photosynthesis is the process by which plants, algae, and some microorganisms use light energy to convert carbon dioxide and water into chemical energy, producing oxygen as a byproduct."),
    ("What is the human heart?",
     "The human heart is a muscular organ that pumps blood through the circulatory system."),
    ("What is the solar system?",
     "The solar system consists of the Sun and the objects gravitationally bound to it, including planets, dwarf planets, moons, asteroids, and comets."),
    ("Why does the Moon have phases?",
     "The Moon appears to change phase because we see different portions of its sunlit half as it orbits Earth."),
    ("What is a galaxy?",
     "A galaxy is a gravitationally bound system containing stars, gas, dust, and dark matter."),
]

for q, a in science:
    for _ in range(250):
        add(q, a, "SCIENCE")

# ---------------- COMPUTER SCIENCE ----------------

cs = [
    ("What is an algorithm?",
     "An algorithm is a finite, precise sequence of steps for solving a problem or performing a computation."),
    ("What is a variable in programming?",
     "A variable is a named storage location associated with a value that a program can read or change."),
    ("What is a function?",
     "A function is a reusable block of code that performs a defined operation and can accept inputs and produce outputs."),
    ("What is a loop?",
     "A loop repeatedly executes a block of instructions while a condition remains true or for a defined sequence of items."),
    ("What is a database?",
     "A database is an organized collection of data that can be stored, queried, and updated."),
    ("What is machine learning?",
     "Machine learning is a field of computing in which models learn statistical patterns from data to perform tasks."),
    ("What is a neural network?",
     "A neural network is a computational model made of connected parameterized units that transform inputs into outputs."),
    ("What is overfitting?",
     "Overfitting occurs when a model learns the training data too specifically and performs poorly on unseen data."),
]

for q, a in cs:
    for _ in range(300):
        add(q, a, "COMPUTER SCIENCE")

# ---------------- GENERAL KNOWLEDGE ----------------

general = [
    ("What is Earth?",
     "Earth is the third planet from the Sun and the largest of the four terrestrial planets in the Solar System."),
    ("What is the Sun?",
     "The Sun is the star at the center of the Solar System and is composed mainly of hydrogen and helium."),
    ("What is water?",
     "Water is the chemical compound H2O, consisting of two hydrogen atoms bonded to one oxygen atom."),
    ("What is the speed of light in vacuum?",
     "The speed of light in vacuum is approximately 299,792,458 meters per second."),
    ("What is the boiling point of water at standard atmospheric pressure?",
     "Water boils at 100 degrees Celsius at standard atmospheric pressure."),
    ("What is the freezing point of water at standard atmospheric pressure?",
     "Water freezes at 0 degrees Celsius at standard atmospheric pressure."),
    ("What is the largest organ of the human body?",
     "The skin is the largest organ of the human body."),
    ("How many continents are commonly taught?",
     "A common geographic convention recognizes seven continents: Africa, Antarctica, Asia, Europe, North America, South America, and Australia."),
]

for q, a in general:
    for _ in range(300):
        add(q, a, "GENERAL KNOWLEDGE")

# ---------------- LOGIC ----------------

logic = [
    ("If all A are B and all B are C, what follows?",
     "All A are C."),
    ("If a statement and its negation cannot both be true, what principle is this?",
     "This expresses the principle of non-contradiction."),
    ("What is a necessary condition?",
     "A necessary condition is something that must be true for another statement or event to be true."),
    ("What is a sufficient condition?",
     "A sufficient condition is something that guarantees another statement or event."),
]

for q, a in logic:
    for _ in range(400):
        add(q, a, "LOGIC")

# ---------------- LANGUAGE / EXPLANATION ----------------

language = [
    ("How should an explanation be structured?",
     "State the answer clearly, define important terms, explain the reasoning, and give an example when useful."),
    ("What should a careful answer do when evidence is uncertain?",
     "It should distinguish known facts from uncertainty and avoid presenting guesses as established facts."),
    ("What makes an answer useful?",
     "A useful answer is relevant to the question, clear about assumptions, and supported by evidence when evidence is needed."),
]

for q, a in language:
    for _ in range(500):
        add(q, a, "REASONING")

random.shuffle(rows)

out.write_text("".join(rows), encoding="utf-8")

print("Created curriculum:", out)
print("Characters:", out.stat().st_size)

# ------------------------------------------------------------
# 2. Add the curriculum to JARVIS's actual corpus
# ------------------------------------------------------------

r = subprocess.run(
    ["python", "main.py", "ingest-file", str(out)],
    text=True,
)
if r.returncode != 0:
    raise RuntimeError("Curriculum ingestion failed.")

print("\nCurriculum ingested successfully.")

# ------------------------------------------------------------
# 3. Train in checkpointed 100-step chunks
#    80 chunks = 8,000 training steps.
#    Each completed chunk saves a new checkpoint.
# ------------------------------------------------------------

for i in range(80):
    print(f"\n========== TRAIN CHUNK {i+1}/80 ==========")

    r = subprocess.run(
        ["python", "main.py", "train", "--steps", "100"],
        text=True,
    )

    if r.returncode != 0:
        print("Training chunk failed; stopping so the latest checkpoint is preserved.")
        break

print("\n========== FINAL BENCHMARK ==========")
subprocess.run(["python", "main.py", "benchmark"], check=False)

print("\n========== FINAL STATUS ==========")
subprocess.run(["python", "main.py", "status"], check=False)

/content/jarvis_v6
Created curriculum: /content/JARVIS_CORE_CURRICULUM.txt
Characters: 4621471

Curriculum ingested successfully.

========== TRAIN CHUNK 1/80 ==========

========== TRAIN CHUNK 2/80 ==========

========== TRAIN CHUNK 3/80 ==========

========== TRAIN CHUNK 4/80 ==========

========== TRAIN CHUNK 5/80 ==========

========== TRAIN CHUNK 6/80 ==========

========== TRAIN CHUNK 7/80 ==========

========== TRAIN CHUNK 8/80 ==========

========== TRAIN CHUNK 9/80 ==========

========== TRAIN CHUNK 10/80 ==========

========== TRAIN CHUNK 11/80 ==========

========== TRAIN CHUNK 12/80 ==========

========== TRAIN CHUNK 13/80 ==========

========== TRAIN CHUNK 14/80 ==========

========== TRAIN CHUNK 15/80 ==========

========== TRAIN CHUNK 16/80 ==========

========== TRAIN CHUNK 17/80 ==========

========== TRAIN CHUNK 18/80 ==========

========== TRAIN CHUNK 19/80 ==========

========== TRAIN CHUNK 20/80 ==========

========== TRAIN CHUNK 21/80 ==========

========== TRAIN 